# 01 — Target Acquisition

Map RelB-dependent HGSOC biomarker genes to ChEMBL targets and rank by druggability.

**Input:** `../data/inputs/relb_gene_signature_352.csv`
**Output:** `../data/processed/target_manifest.csv`

In [ ]:
import sys, yaml
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '..')
from src.acquisition import get_chembl_id_for_gene

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

Path('../data/processed').mkdir(parents=True, exist_ok=True)
print("Ready.")

## Load biomarker pipeline output

Replace `relb_gene_signature_352.csv` with the ranked output from your
`biomarker-discovery-pipeline`. Required column: `gene_symbol`.
Optional columns: `priority`, `survival_hr`, `survival_p`.

In [ ]:
targets_df = pd.read_csv('../data/inputs/relb_gene_signature_352.csv')
print(f"{len(targets_df)} candidate targets loaded")
targets_df

## Map gene symbols → ChEMBL target IDs

In [ ]:
if 'chembl_id' not in targets_df.columns:
    targets_df['chembl_id'] = None

unmapped = targets_df['chembl_id'].isna()
print(f"Mapping {unmapped.sum()} genes to ChEMBL IDs…")

for idx, row in targets_df[unmapped].iterrows():
    targets_df.at[idx, 'chembl_id'] = get_chembl_id_for_gene(row['gene_symbol'])

targets_df = targets_df.dropna(subset=['chembl_id'])
print(f"\n{len(targets_df)} targets successfully mapped")
targets_df[['gene_symbol', 'chembl_id', 'priority']]

## Druggability proxy: count known ChEMBL ligands

Targets with ≥ 100 known ligands have enough data to train a reliable classifier.

In [ ]:
from chembl_webresource_client.new_client import new_client
activity_api = new_client.activity

def count_ligands(chembl_id):
    acts = activity_api.filter(
        target_chembl_id=chembl_id,
        standard_type__in=['IC50', 'Ki'],
        pchembl_value__isnull=False,
    ).only(['molecule_chembl_id'])
    return len({a['molecule_chembl_id'] for a in acts})

# Filter to high+medium priority first to limit API calls (low-priority genes rarely have enough ligands)
targets_df = targets_df[targets_df['priority'].isin(['high', 'medium'])].copy()
print(f'Querying {len(targets_df)} high+medium priority targets for ligand counts…')
targets_df['n_ligands'] = targets_df['chembl_id'].apply(count_ligands)
targets_df = targets_df.sort_values('n_ligands', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if n >= 100 else '#e74c3c' for n in targets_df['n_ligands']]
ax.barh(targets_df['gene_symbol'], targets_df['n_ligands'], color=colors)
ax.axvline(100, color='black', ls='--', label='Min threshold (100)')
ax.set_xlabel('Known ChEMBL ligands')
ax.set_title('Druggability proxy: HGSOC target library')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/druggability_bar.png', dpi=150)
plt.show()

In [ ]:
MIN_LIGANDS = 100
manifest = targets_df[targets_df['n_ligands'] >= MIN_LIGANDS].copy()
manifest.to_csv('../data/processed/target_manifest.csv', index=False)
print(f"Saved {len(manifest)} screeable targets → data/processed/target_manifest.csv")
manifest